# Map and Environment

In [1]:
import os
import pickle

import cartopy.crs as ccrs
import cmocean
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from dotenv import find_dotenv, load_dotenv
from matplotlib import gridspec
from matplotlib.lines import Line2D

from nescafe23_fluctuations import tools

_ = load_dotenv(find_dotenv())

%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'

### load environment data

In [2]:
fn = f"{os.environ['NESCAFE23_DATADIR']}environment_data/map_bathy.nc"
bathy = xr.open_dataset(fn)["elevation"]

fn = f"{os.environ['NESCAFE23_DATADIR']}environment_data/FCTD_SoundSpeed_Colosi_NESCAFE.mat"
fctd = tools.get_fctd(fn=fn, r_smooth=3000)

fn = f"{os.environ['NESCAFE23_DATADIR']}environment_data/dt_global_twosat_phy_l4_20230430_vDT2024.nc"
ssh = xr.open_dataset(fn)

fn = f"{os.environ['NESCAFE23_DATADIR']}environment_data/dataproducts/full_field.nc"
full_field = xr.open_dataset(fn)

fn = f"{os.environ['NESCAFE23_DATADIR']}dataproducts/ray_run_forward_back.pkl"
with open(fn, "rb") as f:
    rays_forward_back = pickle.load(f)
eigs = rays_forward_back["eigs"]

In [3]:
dss = tools.open_receptions()

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 23/23 [00:18<00:00,  1.21it/s]


In [4]:
ssh_slice = (
    ssh["sla"]
    .isel({"time": 0})
    .sel({"latitude": slice(38, 50), "longitude": slice(-70, -50)})
)

- Bathymetry data from GEBCO (will eventually be hi-res map from Brenden)
- FCTD is from Matthew Alford
- sea surface height is from copernicus, global ocean gridded sea surface height.

In [ ]:
plt.rcParams["text.usetex"] = False
plt.rcParams["font.size"] = 8

coords = {
    "SION": [39 + 37.194 / 60, -63 - 39.427 / 60],
    "SIOE": [38 + 45.906 / 60, -62 - 15.626 / 60],
    "PO1": [38.5, -63.2],
}

gs = gridspec.GridSpec(2, 2)

fig = plt.figure(figsize=(6.67, 4))

ax1 = fig.add_subplot(
    gs[:, 0], projection=ccrs.Orthographic(central_longitude=-62, central_latitude=35)
)
ax1.set_extent([-65, -61, 37.5, 40], ccrs.Geodetic())
# ax1.coastlines()

ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, 1])

## MAP
plt.sca(ax1)
# Add gridlines with labels only on bottom and left
gl = ax1.gridlines(
    draw_labels=True,
    dms=False,
    x_inline=False,
    y_inline=False,
    color="#dddddd",
    alpha=0.2,
)
gl.top_labels = False
gl.right_labels = False
gl.bottom_labels = True
gl.left_labels = True

# Add axis labels
ax1.set_xlabel("Longitude", transform=ax1.transAxes)
ax1.set_ylabel("Latitude", transform=ax1.transAxes)

# Bathymetry
bathy_plot = (-bathy).plot(
    cmap=cmocean.cm.deep,
    transform=ccrs.PlateCarree(),
    ax=ax1,
    vmax=7000,
    vmin=500,
    add_colorbar=False,
    rasterized=True,
)  # cbar_kwargs = {'label':'depth [m]'}, alpha=1, rasterized=True)

# cbar = plt.colorbar(bathy_plot, ax=ax1, boundaries=np.linspace(1000,5000,256), label='depth [m]')
# cbar.ax.invert_yaxis()
# cbar.ax.set_yticks(np.arange(1000,5001,1000))

bathy.plot.contour(
    x="lon",
    y="lat",
    transform=ccrs.PlateCarree(),
    ax=ax1,
    colors="k",
    linewidths=0.5,
    alpha=1,
    levels=np.linspace(-5000, -1000, 15),
    linestyles="-",
)

# FCTD
plt.scatter(
    x=fctd["longitude"],
    y=fctd["latitude"],
    c="w",
    transform=ccrs.PlateCarree(),
    s=5,
    lw=0,
    zorder=100,
)
plt.text(
    -64,
    39.25,
    "FCTD",
    transform=ccrs.PlateCarree(),
    bbox=dict(facecolor="white", edgecolor="black", boxstyle="round"),
)

# Sea Surface Height
ssh_slice.plot.contour(
    transform=ccrs.PlateCarree(), levels=[0.25], colors=["#770000"], linewidths=2
)
plt.title("")

# Moorings
for mooring in coords.keys():
    ax1.scatter(
        coords[mooring][1],
        coords[mooring][0],
        transform=ccrs.PlateCarree(),
        c="w",
        edgecolor="k",
        lw=1,
        s=40,
        zorder=1000,
    )
    ax1.text(
        coords[mooring][1] + 0.15,
        coords[mooring][0] + 0.15,
        mooring,
        bbox=dict(facecolor="white", edgecolor="black", boxstyle="round"),
        transform=ccrs.PlateCarree(),
    )

# legend
custom_lines = [Line2D([0], [0], color="#770000", lw=2)]
leg = ax1.legend(custom_lines, ["0.25 m SSH contour"])
frame = leg.get_frame()
frame.set_alpha(1.0)  # or any transparency value 0–1


## SOUND SPEED AND RAYS
plt.sca(ax2)
full_field["c"].assign_coords({"range": full_field.range / 1000 + 35}).plot(
    x="range", cmap=cmocean.cm.thermal, rasterized=True
)

# plot ducted rays
ray_ids = ["-7.0", "7.0", "-6.0", "6.0"]
for ray_id in ray_ids:
    eig_idx = np.where(eigs.ray_id[0] == ray_id)[0][0]
    plt.plot(
        eigs.rs[0][eig_idx, :] / 1000,
        -eigs.zs[0][eig_idx, :],
        zorder=100,
        c="w",
        lw=1,
        alpha=0.7,
    )

# plot moorings
plt.scatter(x=[150] * 60, y=dss[4].actual_depth.values, c="w", lw=0, s=1)
plt.scatter(x=0, y=2300, s=25, lw=1, c="w", edgecolor="k")

# axis formatting
plt.ylim([5000, 0])
plt.xlim([-5, 155])
plt.ylabel("depth [m]")
ax2.set_xticklabels("")
plt.xlabel("")

plt.title("Sound Speed", loc="left")

## BUOYANCY FREQUENCY AND RAYS
plt.sca(ax3)
full_field["N"].assign_coords({"range": full_field.range / 1000 + 35}).plot(
    x="range", cmap=cmocean.cm.haline, rasterized=True
)

# plot ducted rays
ray_ids = ["-7.0", "7.0", "-6.0", "6.0"]
for ray_id in ray_ids:
    eig_idx = np.where(eigs.ray_id[0] == ray_id)[0][0]
    plt.plot(
        eigs.rs[0][eig_idx, :] / 1000,
        -eigs.zs[0][eig_idx, :],
        zorder=100,
        c="w",
        lw=1,
        alpha=0.7,
    )

# plot moorings
plt.scatter(x=[150] * 60, y=dss[4].actual_depth.values, c="w", lw=0, s=1)
plt.scatter(x=0, y=2300, s=25, lw=1, c="w", edgecolor="k")

# axis formatting
plt.ylim([5000, 0])
plt.xlim([-5, 155])
plt.xlabel("range [km]")
plt.ylabel("depth [m]")

plt.title("Buoyancy Frequency", loc="left")

plt.tight_layout()

fig.savefig("../../figures/Map_and_Environment.pdf", dpi=300)